# Sustainability Data Analysis: Our World in Data CO2 Dataset

This notebook analyzes sustainability trends using the CO2 dataset from Our World in Data. We will explore carbon emissions, identify correlations between economic factors and pollution, and clean the data for further analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
file_path = '/content/owid-co2-data.csv'
df = pd.read_csv(file_path)

# Initial display
display(df.head())
print(df.info())

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50411 entries, 0 to 50410
Data columns (total 79 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   country                                    50411 non-null  object 
 1   year                                       50411 non-null  int64  
 2   iso_code                                   42480 non-null  object 
 3   population                                 41167 non-null  float64
 4   gdp                                        15251 non-null  float64
 5   cement_co2                                 29173 non-null  float64
 6   cement_co2_per_capita                      25648 non-null  float64
 7   co2                                        29384 non-null  float64
 8   co2_growth_abs                             27216 non-null  float64
 9   co2_growth_prct                            26239 non-null  float64
 10  co2_including_luc     

## Initial Analysis and Observations

1. **Dataset Scope**: The dataset contains historical CO2 emissions data across various countries and regions, including specialized metrics like methane (`methane`), nitrous oxide (`nitrous_oxide`), and cumulative emissions.
2. **Feature Relations**:
    - `gdp` and `population`: Usually show a strong positive correlation with `co2` emissions.
    - `co2_per_capita`: Allows for a more equitable comparison between large and small nations.
    - `energy_per_capita`: Often relates directly to the industrialization level and carbon footprint of a region.
3. **Missing Values**: Many columns (like `consumption_co2` or specific gas types) have significant null counts, especially for older historical records where data collection was not standardized.

In [2]:
# Data Cleaning: Handling Null Values

# 1. Dropping columns with more than 80% missing data as they may not be useful for general analysis
threshold = len(df) * 0.2
df_cleaned = df.dropna(thresh=threshold, axis=1).copy()

# 2. Filling missing numerical values
# For time-series sustainability data, interpolation is often more accurate than mean imputation
# We group by country (iso_code) to ensure we don't interpolate between different nations
numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns
df_cleaned[numeric_cols] = df_cleaned.groupby('iso_code')[numeric_cols].transform(lambda x: x.interpolate().fillna(method='bfill').fillna(0))

print(f"Missing values after cleaning: {df_cleaned.isnull().sum().sum()}")
display(df_cleaned.describe())

/tmp/ipykernel_641/3567862162.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_cleaned[numeric_cols] = df_cleaned.groupby('iso_code')[numeric_cols].transform(lambda x: x.interpolate().fillna(method='bfill').fillna(0))
/tmp/ipykernel_641/3567862162.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_cleaned[numeric_cols] = df_cleaned.groupby('iso_code')[numeric_cols].transform(lambda x: x.interpolate().fillna(method='bfill').fillna(0))
/tmp/ipykernel_641/3567862162.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_cleaned[numeric_cols] = df_cleaned.groupby('iso_code')[numeric_cols].transform(lambda x: x.interpolate().fillna(method='bfill').fillna(0))
/tmp/ipykernel_641/3567862162.py:11: FutureWarning: 

Missing values after cleaning: 531377


,year,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,co2_including_luc,co2_including_luc_growth_abs,...,share_global_gas_co2,share_global_luc_co2,share_global_oil_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf
count,42480.000000,4.248000e+04,4.248000e+04,42480.000000,42480.000000,42480.000000,42480.000000,42480.000000,42480.000000,42480.000000,...,42480.000000,42480.000000,42480.000000,42480.000000,42480.000000,42480.000000,42480.000000,42480.000000,42480.000000,42480.000000
mean,1923.265654,1.334454e+07,1.029852e+11,1.225429,0.037065,42.554821,0.908459,56.287973,72.231794,1.253952,...,0.530096,0.560624,0.504029,0.485682,0.000477,0.001422,0.002032,0.000065,87.660553,52.557886
std,63.558715,6.792614e+07,7.075752e+11,15.890919,0.098708,327.002153,16.611324,1359.807506,366.377510,28.863609,...,6.183909,2.506832,4.696468,1.986284,0.002389,0.008832,0.011427,0.000471,440.414238,371.901386
min,1750.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,-545.958000,-100.000000,-84.560000,-1723.476000,...,0.000000,-6.966000,0.000000,-0.824000,-0.001000,0.000000,-0.001000,0.000000,-19.725000,0.000000
25%,1878.000000,1.456626e+05,7.903200e+07,0.000000,0.000000,0.015000,0.000000,0.000000,0.269000,-0.117000,...,0.000000,0.000000,0.000000,0.002000,0.000000,0.000000,0.000000,0.000000,0.269000,0.068000
50%,1927.000000,1.523584e+06,3.611432e+09,0.000000,0.000000,0.217500,0.007000,4.585500,7.271000,0.006000,...,0.000000,0.035000,0.005000,0.059000,0.000000,0.000000,0.000000,0.000000,7.577000,0.690000
75%,1976.000000,6.147327e+06,2.593282e+10,0.134000,0.027000,4.152000,0.118000,21.053000,35.762000,0.880250,...,0.000023,0.289000,0.053000,0.259000,0.000000,0.000000,0.001000,0.000000,40.729500,8.515250
max,2024.000000,1.450936e+09,2.696602e+13,828.710000,2.484000,12289.037000,910.106000,180870.000000,11969.620000,1815.360000,...,100.000000,45.639000,100.000000,27.896000,0.053000,0.247000,0.296000,0.011000,14107.007000,13698.566000


## Summary of Actions

1. **Data Import**: Loaded the `owid-co2-data.csv` dataset using pandas.
2. **Exploration**: Identified key features including GDP, population, and various gas emissions. Noted that many metrics are sparse for early historical years.
3. **Null Handling**:
    - Removed columns with excessive missing data (>80%).
    - Used grouped interpolation within countries to preserve local trends, followed by backward filling for early missing years.
4. **Analysis**: Prepared a cleaner version of the dataset ready for correlation analysis and sustainability trend forecasting.